# 📘 Apostila: Trabalho com Dados usando Python

**Temas:** `open()` · Pandas · NumPy · Exemplos e aplicações · Questões conceituais · Exercícios com gabarito

---

## Como usar esta apostila
- Execute as células **em ordem** (`Shift + Enter`). Várias células reaproveitam variáveis e arquivos criados anteriormente.
- Ícones usados: 💡 dica · ⚠️ atenção · 🧪 experimente alterar o código.
- Os arquivos criados (`.txt`, `.csv`, `.json`, `.xlsx`) ficam na mesma pasta do notebook.

## Pré-requisitos
```bash
pip install pandas numpy matplotlib openpyxl
```

## Sumário
1. [Lendo e escrevendo dados com `open()`](#1)
2. [Pandas](#2)
3. [NumPy](#3)
4. [Exemplos de "como usar" e aplicações](#4)
5. [Questões conceituais](#5)
6. [Exercícios](#6)
7. [Gabarito dos exercícios](#7)
8. [Referências](#8)

In [ ]:
import sys
import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("NumPy: ", np.__version__)
print("Pandas:", pd.__version__)

<a id="1"></a>
# 1. Lendo e escrevendo dados com `open()`

Antes de usar bibliotecas especializadas, é fundamental entender como o Python **nativo** lê e grava arquivos.
A função embutida `open()` devolve um *objeto arquivo* que permite ler ou escrever dados.

```python
open(caminho, modo="r", encoding=None)
```

| Modo | Significado | Se o arquivo não existe | Se já existe |
|:---:|---|---|---|
| `"r"` | leitura (padrão) | erro `FileNotFoundError` | lê |
| `"w"` | escrita | cria | **apaga o conteúdo** e reescreve |
| `"a"` | *append* (acrescentar) | cria | adiciona ao final |
| `"x"` | criação exclusiva | cria | erro `FileExistsError` |
| `"b"` | binário (combina: `"rb"`, `"wb"`) | — | — |
| `"+"` | leitura **e** escrita (combina: `"r+"`) | — | — |

### O gerenciador de contexto `with`
Um arquivo aberto ocupa recursos do sistema operacional e precisa ser **fechado**. O bloco `with` faz isso automaticamente,
mesmo que ocorra um erro no meio do caminho:

```python
with open("dados.txt", "r", encoding="utf-8") as arquivo:
    conteudo = arquivo.read()
# aqui o arquivo já foi fechado automaticamente
```

⚠️ Sempre informe `encoding="utf-8"` ao trabalhar com textos em português (acentos, ç, etc.).

### 1.1 Escrevendo um arquivo de texto

In [ ]:
# Modo 'w': cria o arquivo (ou sobrescreve se já existir)
with open("frutas.txt", "w", encoding="utf-8") as arquivo:
    arquivo.write("maçã\n")                       # write() grava UMA string (o '\n' quebra a linha)
    arquivo.write("banana\n")
    arquivo.writelines(["laranja\n", "uva\n"])    # writelines() grava uma lista de strings

print("Arquivo 'frutas.txt' criado com sucesso!")

### 1.2 Lendo o arquivo inteiro com `read()`

In [ ]:
with open("frutas.txt", "r", encoding="utf-8") as arquivo:
    conteudo = arquivo.read()          # devolve todo o conteúdo em uma única string

print(conteudo)
print("Tipo:", type(conteudo), "| Tamanho:", len(conteudo), "caracteres")

### 1.3 `readline()`, `readlines()` e iteração linha a linha

| Método | Retorna | Quando usar |
|---|---|---|
| `read()` | uma string com tudo | arquivos pequenos |
| `readline()` | a próxima linha (string) | ler cabeçalhos, controle fino |
| `readlines()` | lista de strings (todas as linhas) | arquivos pequenos/médios |
| `for linha in arquivo:` | uma linha por vez | **arquivos grandes** (economiza memória) |

In [ ]:
with open("frutas.txt", encoding="utf-8") as f:
    primeira = f.readline()          # lê apenas uma linha (inclui o '\n')
    restantes = f.readlines()        # lê as linhas restantes em uma lista

print("Primeira linha:", repr(primeira))
print("Restantes:", restantes)

# A forma mais eficiente (não carrega tudo na memória): iterar linha a linha
print("\nIterando linha a linha:")
with open("frutas.txt", encoding="utf-8") as f:
    for numero, linha in enumerate(f, start=1):
        print(f"{numero}: {linha.strip()}")   # strip() remove o '\n' e espaços das pontas

### 1.4 Acrescentando dados com o modo `"a"`

In [ ]:
with open("frutas.txt", "a", encoding="utf-8") as f:
    f.write("manga\n")

with open("frutas.txt", encoding="utf-8") as f:
    linhas = [l.strip() for l in f]      # list comprehension: uma lista limpa

print(linhas)
print("Total de frutas:", len(linhas))

### 1.5 Tratando erros comuns

In [ ]:
try:
    with open("arquivo_inexistente.txt") as f:
        f.read()
except FileNotFoundError as e:
    print("Erro tratado:", e)

# Modo 'x' cria o arquivo, mas falha se ele já existir (protege contra sobrescrita acidental)
try:
    with open("frutas.txt", "x") as f:
        f.write("teste")
except FileExistsError:
    print("O arquivo 'frutas.txt' já existe — nada foi sobrescrito.")

### 1.6 Arquivos CSV com o módulo `csv`

CSV (*Comma-Separated Values*) é o formato mais comum para tabelas em texto puro.
Poderíamos usar `linha.split(",")`, mas isso falha quando um campo contém vírgula ou aspas
(ex.: `"Silva, João"`). O módulo `csv` da biblioteca padrão trata esses casos corretamente.

💡 Use `newline=""` ao abrir arquivos CSV — é a recomendação oficial da documentação do Python.

In [ ]:
import csv

alunos = [
    {"nome": "Ana",   "turma": "1A", "nota": 8.5},
    {"nome": "Bruno", "turma": "1A", "nota": 6.0},
    {"nome": "Carla", "turma": "1B", "nota": 9.2},
    {"nome": "Diego", "turma": "1B", "nota": 4.8},
]

# Escrevendo com DictWriter (cada dicionário vira uma linha)
with open("alunos.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=["nome", "turma", "nota"])
    escritor.writeheader()
    escritor.writerows(alunos)

# Lendo com DictReader (cada linha vira um dicionário)
with open("alunos.csv", newline="", encoding="utf-8") as f:
    leitor = csv.DictReader(f)
    for linha in leitor:
        print(linha["nome"], "-", float(linha["nota"]))   # ⚠️ tudo chega como string!

In [ ]:
# csv.reader devolve listas em vez de dicionários
with open("alunos.csv", newline="", encoding="utf-8") as f:
    leitor = csv.reader(f)
    cabecalho = next(leitor)                       # consome a primeira linha
    notas = [float(linha[2]) for linha in leitor]  # converte a 3ª coluna para float

print("Cabeçalho:", cabecalho)
print("Notas:", notas)
print("Média da turma:", sum(notas) / len(notas))

### 1.7 Arquivos JSON

JSON (*JavaScript Object Notation*) é o formato padrão para troca de dados na web e para arquivos de configuração.
Ele mapeia diretamente para dicionários e listas do Python.

| Função | O que faz |
|---|---|
| `json.dump(obj, arquivo)` | grava um objeto Python em arquivo |
| `json.load(arquivo)` | lê um arquivo e devolve o objeto Python |
| `json.dumps(obj)` / `json.loads(texto)` | mesmas operações, mas com **strings** |

In [ ]:
import json

config = {
    "escola": "E.E. Ciências da Natureza",
    "ano": 2026,
    "bimestres": [1, 2, 3, 4],
    "ativa": True,
}

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)   # ensure_ascii=False preserva acentos

with open("config.json", encoding="utf-8") as f:
    lido = json.load(f)

print(lido)
print("Bimestres:", lido["bimestres"])
print("Tipo do objeto lido:", type(lido))

### 1.8 Modo binário (`"rb"` / `"wb"`)

In [ ]:
# Modo binário: útil para imagens, áudio, arquivos compactados etc. Devolve bytes, não str.
with open("frutas.txt", "rb") as f:
    bytes_lidos = f.read(10)          # lê os 10 primeiros bytes

print(bytes_lidos, type(bytes_lidos))
print("Decodificado:", bytes_lidos.decode("utf-8", errors="ignore"))

### 📌 Resumo da Seção 1
- `open(caminho, modo, encoding)` abre arquivos; use **sempre** dentro de `with`.
- Modos: `r` (ler), `w` (escrever/sobrescrever), `a` (acrescentar), `x` (criar exclusivo), `b` (binário).
- Leitura: `read()`, `readline()`, `readlines()` ou iteração `for linha in f`.
- Escrita: `write()` e `writelines()` — lembre-se do `\n`.
- Dados tabulares → módulo `csv`; dados aninhados/configuração → módulo `json`.
- Tudo que vem de um arquivo texto é **string**: converta com `int()`, `float()` etc.

<a id="2"></a>
# 2. Pandas

**Pandas** é a biblioteca padrão para análise de dados tabulares em Python. Pense nela como uma
"planilha programável": você lê tabelas de CSV, Excel, JSON, bancos de dados etc., e as manipula com comandos curtos e rápidos.

Duas estruturas centrais:

| Estrutura | Descrição | Analogia |
|---|---|---|
| `Series` | vetor **unidimensional** com rótulos (índice) | uma coluna da planilha |
| `DataFrame` | tabela **bidimensional** (linhas × colunas) | a planilha inteira |

Convenção universal: `import pandas as pd`.

### 2.1 Series

In [ ]:
import pandas as pd

temperaturas = pd.Series([22.5, 25.1, 19.8, 30.2],
                         index=["seg", "ter", "qua", "qui"],
                         name="temp_C")
print(temperaturas)
print("\nMédia:", temperaturas.mean())
print("Dia mais quente:", temperaturas.idxmax())
print("Acesso pelo rótulo:", temperaturas["ter"])
print("Acesso pela posição:", temperaturas.iloc[1])

### 2.2 Criando um DataFrame a partir de um dicionário

In [ ]:
dados = {
    "nome":   ["Ana", "Bruno", "Carla", "Diego", "Elisa", "Fábio"],
    "turma":  ["1A", "1A", "1B", "1B", "1C", "1C"],
    "idade":  [15, 16, 15, 17, 16, 15],
    "nota_1": [8.5, 6.0, 9.2, 4.8, 7.5, 5.5],
    "nota_2": [7.0, 6.5, 8.8, 5.2, None, 6.0],     # None vira NaN (valor ausente)
}
df = pd.DataFrame(dados)
df          # a última expressão da célula é exibida como tabela formatada

### 2.3 Inspecionando os dados
Sempre comece uma análise conhecendo o formato, os tipos e as estatísticas básicas.

In [ ]:
print("Formato (linhas, colunas):", df.shape)
print("Colunas:", list(df.columns))
print("\nTipos de dados:")
print(df.dtypes)
print("\nInformações gerais:")
df.info()

In [ ]:
df.describe()   # estatísticas descritivas das colunas numéricas

### 2.4 Lendo arquivos externos
O pandas lê dezenas de formatos: `pd.read_csv`, `pd.read_excel`, `pd.read_json`, `pd.read_sql`, `pd.read_html`...
Vamos ler o CSV criado na Seção 1.

In [ ]:
alunos_csv = pd.read_csv("alunos.csv")
print(alunos_csv.dtypes)      # 💡 repare que 'nota' já veio como float, sem conversão manual
alunos_csv.head()             # head(n) mostra as n primeiras linhas (padrão 5)

### 2.5 Selecionando colunas e linhas

In [ ]:
print(df["nome"])                 # uma coluna  -> Series
print()
print(df[["nome", "nota_1"]])     # lista de colunas -> DataFrame

In [ ]:
# loc  -> por RÓTULO (nome do índice / coluna)
# iloc -> por POSIÇÃO (número inteiro, começando em 0)
print(df.loc[0, "nome"])                 # linha de rótulo 0, coluna 'nome'
print(df.iloc[0, 0])                     # primeira linha, primeira coluna
print()
print(df.loc[1:3, ["nome", "turma"]])    # ⚠️ loc INCLUI o fim do intervalo (1, 2, 3)
print()
print(df.iloc[1:3, 0:2])                 # iloc EXCLUI o fim (1, 2), como fatias de listas

### 2.6 Filtrando com condições (máscaras booleanas)

In [ ]:
aprovados_n1 = df[df["nota_1"] >= 6]
print(aprovados_n1)
print()

# Combinação de condições: & (e), | (ou), ~ (não) — sempre com parênteses
print(df[(df["turma"] == "1B") & (df["idade"] >= 16)])
print()
print(df.query("nota_1 > 7 and idade < 16"))       # sintaxe alternativa, estilo SQL
print()
print(df[df["turma"].isin(["1A", "1C"])])           # pertence a uma lista de valores

### 2.7 Lidando com valores ausentes (`NaN`)
Dados reais quase sempre têm lacunas. Duas estratégias: **remover** (`dropna`) ou **preencher** (`fillna`).

In [ ]:
print("Valores ausentes por coluna:")
print(df.isna().sum())
print()
print("Linhas com algum valor ausente:")
print(df[df.isna().any(axis=1)])

sem_nulos = df.dropna()
print("\nApós dropna:", sem_nulos.shape, "(perdemos uma linha inteira)")

media_n2 = df["nota_2"].mean()                       # a média IGNORA NaN automaticamente
df["nota_2"] = df["nota_2"].fillna(round(media_n2, 1))
print("\nnota_2 preenchida com a média:", round(media_n2, 1))
df

### 2.8 Criando novas colunas

In [ ]:
df["media"] = (df["nota_1"] + df["nota_2"]) / 2                 # operação vetorizada
df["situacao"] = df["media"].apply(lambda m: "Aprovado" if m >= 6 else "Recuperação")
df["nome_maiusculo"] = df["nome"].str.upper()                   # métodos de texto via .str
df

### 2.9 Ordenando

In [ ]:
print(df.sort_values("media", ascending=False)[["nome", "media"]])
print()
print(df.sort_values(["turma", "media"], ascending=[True, False])[["turma", "nome", "media"]])

### 2.10 Agrupando com `groupby`
O padrão **dividir → aplicar → combinar**: separa as linhas por categoria, aplica uma função em cada grupo e junta os resultados.

In [ ]:
print(df.groupby("turma")["media"].mean())
print()

resumo = df.groupby("turma").agg(
    qtd_alunos=("nome", "count"),
    media_turma=("media", "mean"),
    maior_nota=("media", "max"),
    idade_media=("idade", "mean"),
)
resumo

In [ ]:
print(df["situacao"].value_counts())               # contagem de cada categoria
print()
print("Turmas distintas:", df["turma"].unique())
print("Quantidade de turmas:", df["turma"].nunique())

### 2.11 Combinando tabelas: `merge` e `concat`
- `merge` junta tabelas **lado a lado** por uma chave comum (como o *JOIN* do SQL ou o PROCV do Excel).
- `concat` **empilha** tabelas (linhas embaixo de linhas, ou colunas ao lado de colunas).

In [ ]:
professores = pd.DataFrame({
    "turma": ["1A", "1B", "1C"],
    "professor": ["Prof. Marcos", "Profa. Lúcia", "Prof. Tadeu"],
})
completo = pd.merge(df, professores, on="turma", how="left")
print(completo[["nome", "turma", "professor"]])

novos = pd.DataFrame({"nome": ["Gabi"], "turma": ["1A"], "idade": [16], "nota_1": [9.0], "nota_2": [8.0]})
empilhado = pd.concat([df, novos], ignore_index=True)
print("\nApós concat:", empilhado.shape)
empilhado.tail(3)     # colunas que 'novos' não tinha ficam como NaN

### 2.12 Tabelas dinâmicas com `pivot_table`

In [ ]:
vendas = pd.DataFrame({
    "mes":     ["Jan", "Jan", "Fev", "Fev", "Mar", "Mar"],
    "produto": ["Caderno", "Caneta", "Caderno", "Caneta", "Caderno", "Caneta"],
    "valor":   [1200, 300, 950, 420, 1500, 380],
})
tabela = vendas.pivot_table(index="mes", columns="produto", values="valor", aggfunc="sum")
tabela

### 2.13 Trabalhando com datas

In [ ]:
serie_tempo = pd.DataFrame({
    "data": pd.date_range("2026-03-01", periods=6, freq="D"),
    "temperatura": [24.1, 25.3, 23.8, 26.0, 27.2, 22.9],
})
serie_tempo["dia_semana"] = serie_tempo["data"].dt.day_name()
serie_tempo["mes"] = serie_tempo["data"].dt.month
serie_tempo["variacao"] = serie_tempo["temperatura"].diff()     # diferença em relação ao dia anterior
serie_tempo

### 2.14 Salvando resultados

In [ ]:
df.to_csv("alunos_processado.csv", index=False)      # index=False evita gravar a coluna de índice
df.to_json("alunos_processado.json", orient="records", force_ascii=False, indent=2)

try:
    df.to_excel("alunos_processado.xlsx", index=False)   # requer o pacote openpyxl
    print("Excel salvo!")
except ModuleNotFoundError:
    print("Instale openpyxl para salvar em Excel: pip install openpyxl")

import os
print([arq for arq in os.listdir(".") if arq.startswith("alunos_processado")])

### 📌 Resumo da Seção 2

| Tarefa | Comando |
|---|---|
| Ler / salvar | `pd.read_csv`, `pd.read_excel`, `df.to_csv`, `df.to_excel`, `df.to_json` |
| Inspecionar | `head`, `tail`, `shape`, `info`, `describe`, `dtypes` |
| Selecionar | `df["col"]`, `df[["a","b"]]`, `df.loc[...]`, `df.iloc[...]` |
| Filtrar | `df[condição]`, `query`, `isin` |
| Ausentes | `isna`, `dropna`, `fillna` |
| Transformar | `apply`, `map`, `.str`, `.dt`, operações vetorizadas |
| Agregar | `groupby`, `agg`, `value_counts`, `pivot_table` |
| Combinar | `merge`, `concat` |
| Ordenar | `sort_values`, `nlargest`, `nsmallest` |

<a id="3"></a>
# 3. NumPy em Python

**NumPy** (*Numerical Python*) é a base de praticamente todo o ecossistema científico do Python — o próprio pandas é construído sobre ele.
Sua estrutura central é o `ndarray`: um vetor/matriz **homogêneo** (todos os elementos do mesmo tipo) armazenado de forma contígua na memória.

Por que usar NumPy em vez de listas?
1. **Velocidade** — as operações rodam em código C otimizado, dezenas de vezes mais rápidas.
2. **Vetorização** — você escreve `a * 2` em vez de um laço `for`.
3. **Broadcasting** — operações entre arrays de formatos diferentes, de forma automática.
4. **Funções matemáticas prontas** — estatística, álgebra linear, números aleatórios, transformadas.

Convenção universal: `import numpy as np`.

### 3.1 Criando arrays e conhecendo seus atributos

In [ ]:
import numpy as np

lista = [1, 2, 3, 4, 5]
arr = np.array(lista)
print(arr, type(arr))
print("dtype:", arr.dtype, "| shape:", arr.shape, "| ndim:", arr.ndim, "| size:", arr.size)

matriz = np.array([[1, 2, 3], [4, 5, 6]])
print(matriz)
print("shape:", matriz.shape, "| ndim:", matriz.ndim)

decimais = np.array([1, 2, 3], dtype=np.float64)     # forçando o tipo
print(decimais, decimais.dtype)

### 3.2 Funções geradoras

In [ ]:
print(np.arange(0, 10, 2))          # início, fim (exclusivo), passo
print(np.linspace(0, 1, 5))         # 5 valores igualmente espaçados entre 0 e 1 (inclusive)
print(np.zeros((2, 3)))
print(np.ones(4))
print(np.eye(3))                    # matriz identidade
print(np.full((2, 2), 7))

rng = np.random.default_rng(seed=42)   # gerador com semente -> resultados reprodutíveis
print(rng.integers(1, 7, size=10))     # 10 lançamentos de um dado
print(rng.normal(loc=0, scale=1, size=3).round(3))   # distribuição normal

### 3.3 Indexação, fatiamento e máscaras booleanas

In [ ]:
a = np.arange(10, 20)
print(a)
print(a[0], a[-1])
print(a[2:5])
print(a[::2])                 # de 2 em 2

m = np.arange(1, 13).reshape(3, 4)
print(m)
print("Elemento linha 1, coluna 2:", m[1, 2])
print("Segunda linha:", m[1])
print("Terceira coluna:", m[:, 2])
print("Submatriz:", m[0:2, 1:3], sep="\n")

# Máscara booleana: filtra pelos elementos onde a condição é True
print("Pares:", a[a % 2 == 0])
print("Maiores que 15:", a[a > 15])

### 3.4 Operações vetorizadas e broadcasting

In [ ]:
x = np.array([1, 2, 3, 4])
y = np.array([10, 20, 30, 40])
print(x + y, x * y, y / x, x ** 2, sep="\n")     # elemento a elemento, sem laço for
print(np.sqrt(y), np.exp(x).round(2), sep="\n")

# Broadcasting: o escalar é "esticado" para o formato do array
print(x * 10)
celsius = np.array([0, 25, 37, 100])
fahrenheit = celsius * 9 / 5 + 32
print("Fahrenheit:", fahrenheit)

# Broadcasting entre formatos compatíveis: (3,1) + (1,4) -> (3,4)
col = np.array([[1], [2], [3]])
lin = np.array([[10, 20, 30, 40]])
print(col + lin)

### 3.5 Estatística descritiva e o parâmetro `axis`
- `axis=0` → percorre as **linhas** (resultado por coluna).
- `axis=1` → percorre as **colunas** (resultado por linha).

In [ ]:
notas = np.array([[8.5, 7.0], [6.0, 6.5], [9.2, 8.8], [4.8, 5.2], [7.5, 6.9], [5.5, 6.0]])
print("Média geral:", notas.mean().round(3))
print("Média por coluna (axis=0):", notas.mean(axis=0).round(3))   # média de cada prova
print("Média por linha (axis=1):", notas.mean(axis=1).round(3))    # média de cada aluno
print("Desvio padrão:", notas.std().round(3))
print("Mínimo e máximo:", notas.min(), notas.max())
print("Índice do maior valor (array achatado):", notas.argmax())
print("Soma acumulada:", np.cumsum([1, 2, 3, 4]))
print("Mediana:", np.median(notas))
print("Percentil 75:", np.percentile(notas, 75))

### 3.6 Remodelando e combinando arrays

In [ ]:
v = np.arange(12)
print(v.reshape(3, 4))
print(v.reshape(3, 4).T)              # transposta
print(v.reshape(2, -1))               # -1 = "calcule essa dimensão para mim"

a1 = np.array([1, 2, 3]); a2 = np.array([4, 5, 6])
print(np.concatenate([a1, a2]))
print(np.vstack([a1, a2]))            # empilha verticalmente
print(np.hstack([a1, a2]))            # empilha horizontalmente
print(np.where(a1 > 1, "alto", "baixo"))   # if/else vetorizado

### 3.7 Comparando desempenho: lista Python × NumPy

In [ ]:
import time

n = 1_000_000
lista_py = list(range(n))
arr_np = np.arange(n)

inicio = time.perf_counter()
soma_lista = sum(v * 2 for v in lista_py)
t_lista = time.perf_counter() - inicio

inicio = time.perf_counter()
soma_np = (arr_np * 2).sum()
t_np = time.perf_counter() - inicio

print(f"Lista Python: {t_lista:.4f} s")
print(f"NumPy:        {t_np:.4f} s")
print(f"NumPy foi ~{t_lista / t_np:.0f}x mais rápido (resultados iguais: {soma_lista == soma_np})")

### 3.8 Álgebra linear básica

In [ ]:
A = np.array([[2, 1], [1, 3]])
b = np.array([5, 10])

print("A @ A =\n", A @ A)                 # multiplicação de matrizes (@ ou np.matmul)
print("Determinante:", np.linalg.det(A))
print("Inversa:\n", np.linalg.inv(A))
x = np.linalg.solve(A, b)                 # resolve o sistema A·x = b
print("Solução de {2x + y = 5 ; x + 3y = 10}:", x)

### 3.9 NumPy + pandas trabalhando juntos
Um DataFrame é, por baixo, um conjunto de arrays NumPy. Funções do NumPy aceitam colunas do pandas diretamente.

In [ ]:
df_np = pd.DataFrame(notas, columns=["prova_1", "prova_2"])
df_np["media"] = df_np.mean(axis=1)
df_np["conceito"] = np.where(df_np["media"] >= 7, "B",
                    np.where(df_np["media"] >= 5, "R", "I"))
print(df_np)
print("\nDataFrame -> array NumPy:", type(df_np[["prova_1", "prova_2"]].to_numpy()))
print("Correlação entre as provas:", np.corrcoef(df_np["prova_1"], df_np["prova_2"])[0, 1].round(3))

### 📌 Resumo da Seção 3

| Tarefa | Comando |
|---|---|
| Criar | `np.array`, `arange`, `linspace`, `zeros`, `ones`, `eye`, `full`, `default_rng()` |
| Atributos | `shape`, `ndim`, `size`, `dtype` |
| Acessar | `a[i]`, `a[i, j]`, `a[ini:fim]`, `a[:, j]`, `a[mascara]` |
| Operar | `+ - * / **`, `np.sqrt`, `np.exp`, `np.log`, broadcasting |
| Estatística | `mean`, `std`, `min`, `max`, `sum`, `median`, `percentile`, `argmax`, `cumsum` |
| Remodelar | `reshape`, `.T`, `concatenate`, `vstack`, `hstack`, `np.where` |
| Álgebra linear | `@`, `np.linalg.det`, `inv`, `solve` |

<a id="4"></a>
# 4. Exemplos de "como usar" e aplicações

Nesta seção juntamos as três ferramentas em situações realistas. Cada aplicação segue o fluxo típico de um projeto de dados:

**obter → limpar → transformar → analisar → comunicar**

## 4.1 Aplicação: Boletim escolar
Um professor registrou as notas de 3 bimestres e as faltas dos alunos. Vamos gerar o arquivo com `csv`, ler com pandas e produzir um diagnóstico.

In [ ]:
import csv
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
nomes = ["Ana", "Bruno", "Carla", "Diego", "Elisa", "Fábio", "Gabi", "Hugo", "Iara", "João", "Karen", "Lucas"]
turmas = ["1A"] * 4 + ["1B"] * 4 + ["1C"] * 4

# Etapa 1: gerar o arquivo bruto com open() + csv
with open("boletim.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["nome", "turma", "bim1", "bim2", "bim3", "faltas"])
    for nome, turma in zip(nomes, turmas):
        notas_aluno = np.clip(rng.normal(6.5, 1.8, size=3), 0, 10).round(1)   # notas entre 0 e 10
        w.writerow([nome, turma, *notas_aluno, int(rng.integers(0, 12))])

# Etapa 2: ler com pandas
boletim = pd.read_csv("boletim.csv")
boletim

In [ ]:
# Etapa 3: transformar
boletim["media"] = boletim[["bim1", "bim2", "bim3"]].mean(axis=1).round(2)
boletim["evolucao"] = boletim["bim3"] - boletim["bim1"]

def classificar(linha):
    if linha["faltas"] > 8:
        return "Reprovado por falta"
    if linha["media"] >= 6:
        return "Aprovado"
    return "Recuperação"

boletim["situacao"] = boletim.apply(classificar, axis=1)     # axis=1: função recebe cada LINHA
boletim.sort_values("media", ascending=False)

In [ ]:
# Etapa 4: analisar
print("Situação geral:")
print(boletim["situacao"].value_counts(), "\n")

print("Média por turma e bimestre:")
print(boletim.groupby("turma")[["bim1", "bim2", "bim3", "media"]].mean().round(2), "\n")

print("Aluno que mais evoluiu:", boletim.loc[boletim["evolucao"].idxmax(), "nome"])
print("Alunos em recuperação:", ", ".join(boletim.loc[boletim["situacao"] == "Recuperação", "nome"]))

In [ ]:
# Etapa 5: comunicar (gráfico rápido com matplotlib)
import matplotlib.pyplot as plt

medias_turma = boletim.groupby("turma")[["bim1", "bim2", "bim3"]].mean()
medias_turma.T.plot(marker="o", figsize=(7, 4), title="Evolução da média por turma")
plt.xlabel("Bimestre"); plt.ylabel("Média"); plt.ylim(0, 10); plt.grid(alpha=0.3)
plt.show()

## 4.2 Aplicação: Registro de vendas
Uma papelaria registra cada venda. Queremos faturamento por produto, por mês e o melhor dia.

In [ ]:
rng = np.random.default_rng(10)
datas = pd.date_range("2026-01-01", periods=90, freq="D")

vendas = pd.DataFrame({
    "data": rng.choice(datas, size=300),
    "produto": rng.choice(["Caderno", "Caneta", "Mochila", "Lápis"], size=300, p=[0.3, 0.35, 0.1, 0.25]),
    "quantidade": rng.integers(1, 6, size=300),
})
precos = {"Caderno": 18.9, "Caneta": 2.5, "Mochila": 129.0, "Lápis": 1.2}
vendas["preco_unit"] = vendas["produto"].map(precos)          # map com dicionário = "PROCV"
vendas["total"] = vendas["quantidade"] * vendas["preco_unit"]
vendas["mes"] = vendas["data"].dt.to_period("M").astype(str)  # ex.: '2026-01'
vendas.head()

In [ ]:
print("Faturamento total: R$", round(vendas["total"].sum(), 2))

print("\nFaturamento por produto:")
print(vendas.groupby("produto")["total"].sum().sort_values(ascending=False).round(2))

print("\nFaturamento por mês e produto:")
print(vendas.pivot_table(index="mes", columns="produto", values="total", aggfunc="sum").round(2))

print("\nTicket médio por venda: R$", round(vendas["total"].mean(), 2))
top_dia = vendas.groupby("data")["total"].sum().idxmax()
print("Dia de maior faturamento:", top_dia.date())

## 4.3 Aplicação: Dados de sensor (estação meteorológica) com NumPy
Um sensor registrou a temperatura a cada hora durante uma semana, mas houve falhas de leitura.
Vamos detectar *outliers* com z-score, limpar, suavizar com média móvel e resumir por dia.

In [ ]:
rng = np.random.default_rng(3)
horas = np.arange(0, 24 * 7)                                   # uma semana, hora a hora (168 leituras)
temp = 22 + 6 * np.sin((horas - 8) / 24 * 2 * np.pi) + rng.normal(0, 0.8, size=horas.size)
temp[50] = 45.0     # falha de sensor (valor absurdo)
temp[120] = -5.0

# Detecção de outliers por z-score
media, desvio = temp.mean(), temp.std()
z = (temp - media) / desvio
outliers = np.where(np.abs(z) > 3)[0]
print("Leituras suspeitas nas horas:", outliers, "-> valores:", temp[outliers].round(1))

temp_limpa = temp.copy()
temp_limpa[outliers] = np.nan
print("Média bruta: %.2f °C | Média limpa: %.2f °C" % (temp.mean(), np.nanmean(temp_limpa)))

# Interpolação dos buracos e média móvel de 6 h (pandas lida muito bem com NaN)
serie = pd.Series(temp_limpa).interpolate()
media_movel = serie.rolling(window=6, center=True).mean()

# Estatísticas por dia: reshape (7 dias × 24 horas)
por_dia = serie.to_numpy().reshape(7, 24)
print("Máxima por dia:", por_dia.max(axis=1).round(1))
print("Mínima por dia:", por_dia.min(axis=1).round(1))
print("Amplitude térmica média:", (por_dia.max(axis=1) - por_dia.min(axis=1)).mean().round(2), "°C")

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(horas, temp, alpha=0.4, label="bruto (com falhas)")
plt.plot(horas, media_movel, linewidth=2, label="média móvel 6 h (limpo)")
plt.scatter(outliers, temp[outliers], color="red", zorder=3, label="outliers")
plt.title("Temperatura horária — uma semana"); plt.xlabel("hora"); plt.ylabel("°C")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 4.4 Aplicação: Pipeline completo — `open()` → limpeza → pandas → relatório
Na vida real os dados chegam "sujos": separador `;`, vírgula decimal, espaços extras, linhas vazias e marcadores como `N/A`.

In [ ]:
# 1) Um arquivo "sujo", como frequentemente chega na vida real
bruto = """nome;turma;nota
Ana ; 1A ; 8,5
Bruno;1A;6,0
Carla;1B;9,2

Diego;1B;N/A
Elisa;1C;7,5
"""
with open("notas_brutas.txt", "w", encoding="utf-8") as f:
    f.write(bruto)

# 2) Limpeza com open() + Python puro
registros = []
with open("notas_brutas.txt", encoding="utf-8") as f:
    next(f)                                    # pula o cabeçalho
    for linha in f:
        linha = linha.strip()
        if not linha:                          # ignora linhas vazias
            continue
        nome, turma, nota = [campo.strip() for campo in linha.split(";")]
        nota = nota.replace(",", ".")
        registros.append({"nome": nome, "turma": turma,
                          "nota": float(nota) if nota != "N/A" else np.nan})

# 3) Análise com pandas
limpo = pd.DataFrame(registros)
print(limpo, "\n")
print("Média por turma (ignorando ausentes):")
print(limpo.groupby("turma")["nota"].mean().round(2))

# 4) Relatório final em texto, usando open()
with open("relatorio.txt", "w", encoding="utf-8") as f:
    f.write("RELATÓRIO DE NOTAS\n==================\n")
    f.write(f"Alunos: {len(limpo)} | Com nota: {limpo['nota'].notna().sum()}\n")
    f.write(f"Média geral: {limpo['nota'].mean():.2f}\n")
    for turma, media_t in limpo.groupby("turma")["nota"].mean().items():
        f.write(f"  Turma {turma}: {media_t:.2f}\n")

print("\n--- relatorio.txt ---")
with open("relatorio.txt", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# 💡 O pandas consegue fazer boa parte dessa limpeza em uma única linha:
atalho = pd.read_csv("notas_brutas.txt", sep=";", decimal=",", na_values=["N/A"], skipinitialspace=True)
atalho["nome"] = atalho["nome"].str.strip()
atalho["turma"] = atalho["turma"].str.strip()
atalho

### 📌 Quando usar cada ferramenta?

| Situação | Ferramenta recomendada |
|---|---|
| Arquivo de texto simples, log, configuração | `open()` (+ `csv` / `json`) |
| Formato exótico que precisa de limpeza linha a linha | `open()` para limpar → pandas para analisar |
| Tabela com colunas de tipos diferentes (texto, número, data) | **pandas** |
| Agrupar, juntar tabelas, tabelas dinâmicas, datas | **pandas** |
| Cálculo numérico pesado, matrizes, simulações, álgebra linear | **NumPy** |
| Filtros condicionais rápidos em colunas numéricas | NumPy (`np.where`) dentro do pandas |

<a id="5"></a>
# 5. Questões conceituais

Responda com suas palavras antes de conferir as respostas.

**Q1.** Por que é recomendado usar `with open(...)` em vez de `arquivo = open(...)` seguido de `arquivo.close()`?

**Q2.** Qual é a diferença entre os modos `"w"`, `"a"` e `"x"`? Em que situação `"x"` é mais seguro?

**Q3.** Por que devemos especificar `encoding="utf-8"` ao abrir arquivos de texto em português? O que pode acontecer se não fizermos isso?

**Q4.** Por que não é recomendado ler um arquivo CSV apenas com `linha.split(",")`? O que o módulo `csv` resolve?

**Q5.** Compare `read()`, `readline()` e `readlines()`. Qual estratégia é a mais adequada para um arquivo de 10 GB?

**Q6.** Qual é a diferença entre uma `Series` e um `DataFrame` no pandas?

**Q7.** Explique a diferença entre `df.loc[...]` e `df.iloc[...]`. O que acontece com o fim do intervalo em cada um?

**Q8.** O que acontece internamente quando executamos `df[df["nota"] >= 6]`?

**Q9.** Explique o padrão *split-apply-combine* implementado por `groupby`.

**Q10.** Por que o NumPy é muito mais rápido que listas Python para operações numéricas?

**Q11.** O que é *broadcasting*? Dê um exemplo em que ele funciona e um em que ele falha.

**Q12.** O que significam `axis=0` e `axis=1` em `array.mean(axis=...)`?

**Q13.** Quando é melhor usar `dropna()` e quando é melhor usar `fillna()`? Quais os riscos de cada escolha?

**Q14.** Qual é a diferença entre `pd.merge` e `pd.concat`?

**Q15.** O que é `NaN`? Como `mean()` do pandas o trata? E `np.mean` de um array com `NaN`?

<details>
<summary><b>👁️ Clique para ver as respostas das questões conceituais</b></summary>

**R1.** O `with` garante que o arquivo seja fechado automaticamente ao sair do bloco, **mesmo se ocorrer uma exceção**. Com `close()` manual, um erro antes da chamada deixaria o arquivo aberto (recursos presos, dados não gravados no disco).

**R2.** `"w"` cria o arquivo ou **apaga** o conteúdo existente; `"a"` cria ou **acrescenta** ao final, preservando o conteúdo; `"x"` cria o arquivo e **lança `FileExistsError`** se ele já existir. `"x"` é mais seguro quando não queremos correr o risco de sobrescrever dados importantes.

**R3.** A codificação padrão depende do sistema operacional (no Windows costuma ser `cp1252`). Sem especificar, um arquivo gravado em um sistema pode ser lido de forma errada em outro, gerando caracteres estranhos (`Ã§`, `Ã£`) ou `UnicodeDecodeError`. UTF-8 é o padrão universal.

**R4.** `split(",")` quebra incorretamente campos que contêm vírgula dentro de aspas (`"Silva, João"`), não trata aspas escapadas nem quebras de linha dentro de campos. O módulo `csv` implementa o padrão RFC 4180 e resolve esses casos, além de permitir trocar separador, aspas etc.

**R5.** `read()` devolve tudo em uma string; `readline()` devolve uma linha por chamada; `readlines()` devolve uma lista com todas as linhas. Para um arquivo de 10 GB, `read()` e `readlines()` estourariam a memória — o correto é **iterar linha a linha** (`for linha in arquivo`), que carrega uma linha por vez.

**R6.** `Series` é unidimensional (um vetor com índice, como uma coluna). `DataFrame` é bidimensional (várias `Series` compartilhando o mesmo índice, como uma planilha). Selecionar uma coluna de um `DataFrame` devolve uma `Series`.

**R7.** `loc` seleciona por **rótulo** (nome do índice/coluna) e **inclui** o fim do intervalo; `iloc` seleciona por **posição inteira** (0, 1, 2...) e **exclui** o fim, como as fatias de listas do Python.

**R8.** `df["nota"] >= 6` gera uma `Series` booleana (True/False para cada linha) — a **máscara**. Ao passá-la em `df[...]`, o pandas devolve apenas as linhas em que a máscara é `True`.

**R9.** *Split*: as linhas são divididas em grupos conforme os valores da(s) coluna(s) chave. *Apply*: uma função (média, soma, contagem...) é aplicada a cada grupo separadamente. *Combine*: os resultados são reunidos em uma nova estrutura indexada pelos grupos.

**R10.** Arrays NumPy são homogêneos e armazenados em blocos contíguos de memória, permitindo que as operações sejam executadas em laços compilados em C (vetorização), sem a sobrecarga de interpretar cada objeto Python. Listas guardam ponteiros para objetos heterogêneos, exigindo verificação de tipo a cada elemento.

**R11.** *Broadcasting* é a regra que permite operar arrays de formatos diferentes "esticando" virtualmente as dimensões de tamanho 1 (ou ausentes). Funciona: `(3,1) + (1,4) → (3,4)` ou `array * escalar`. Falha: `(3,) + (4,)` — as dimensões não são iguais nem uma delas é 1 → `ValueError`.

**R12.** `axis=0` percorre **ao longo das linhas** (colapsa as linhas → um resultado por coluna). `axis=1` percorre **ao longo das colunas** (colapsa as colunas → um resultado por linha). Em uma matriz de alunos × provas: `axis=0` dá a média de cada prova; `axis=1` dá a média de cada aluno.

**R13.** `dropna()` quando os ausentes são poucos e a linha inteira perde sentido sem o valor (risco: perder dados e enviesar a amostra). `fillna()` quando queremos preservar as linhas, usando média, mediana, zero, valor anterior etc. (risco: inventar dados e distorcer estatísticas, especialmente variância).

**R14.** `merge` junta tabelas **lado a lado** por uma ou mais colunas-chave em comum (como o JOIN do SQL), com opções `inner/left/right/outer`. `concat` **empilha** tabelas (verticalmente, por padrão) alinhando pelas colunas, sem usar chave.

**R15.** `NaN` (*Not a Number*) é o marcador de valor ausente em ponto flutuante. `Series.mean()` do pandas **ignora** os `NaN` por padrão (`skipna=True`). Já `np.mean` de um array com `NaN` devolve `nan`; para ignorar, usa-se `np.nanmean`.

</details>

<a id="6"></a>
# 6. Exercícios

Resolva cada exercício na célula indicada. O gabarito completo está na Seção 7.

⚠️ Os exercícios dos blocos B, C e D usam arquivos e variáveis criados na Seção 4 (`boletim.csv`, `vendas`, `serie`, `notas_brutas.txt`). Execute a Seção 4 antes.

---
## Bloco A — `open()`, `csv` e `json`

**Exercício 1.** Crie um arquivo `compras.txt` com 5 itens de uma lista de compras (um por linha). Em seguida, leia o arquivo e imprima os itens numerados (`1. arroz`, `2. feijão`...).

In [ ]:
# Sua resposta — Exercício 1

**Exercício 2.** Grave um texto de 3 ou mais linhas em `texto.txt`. Depois, lendo o arquivo **linha a linha**, conte quantas linhas, palavras e caracteres ele possui (como o comando `wc` do Linux).

In [ ]:
# Sua resposta — Exercício 2

**Exercício 3.** Usando `csv.DictWriter`, crie `estados.csv` com as colunas `estado`, `capital`, `populacao` e pelo menos 4 linhas. Em seguida, leia o arquivo com `csv.DictReader` e imprima o estado mais populoso. Lembre-se: os valores lidos são strings!

In [ ]:
# Sua resposta — Exercício 3

**Exercício 4.** Salve um dicionário de preferências (`tema`, `fonte`, `notificacoes`) em `prefs.json`. Leia o arquivo de volta, altere o tema para `"escuro"`, aumente a fonte em 2 pontos e salve novamente. Exiba o conteúdo final do arquivo.

In [ ]:
# Sua resposta — Exercício 4

---
## Bloco B — Pandas

**Exercício 5.** A partir de `boletim.csv`, responda: **(a)** quantos alunos há em cada turma? **(b)** qual a média do `bim2` na turma 1B? **(c)** quais são os 3 alunos com maior média nos três bimestres?

In [ ]:
# Sua resposta — Exercício 5

**Exercício 6.** Crie a coluna `faixa_faltas` com as categorias **Baixa** (0–3 faltas), **Média** (4–8) e **Alta** (mais de 8). Dica: `pd.cut(...)` ou `apply`. Depois conte quantos alunos há em cada faixa.

In [ ]:
# Sua resposta — Exercício 6

**Exercício 7.** Com o DataFrame `vendas` (Seção 4.2): **(a)** qual produto teve a maior **quantidade** vendida (não faturamento)? **(b)** qual é o faturamento total por dia da semana (segunda, terça...)? Dica: `.dt.dayofweek`.

In [ ]:
# Sua resposta — Exercício 7

**Exercício 8.** Crie um DataFrame `professores` com as colunas `turma` e `professor` (1A → Prof. Marcos, 1B → Profa. Lúcia, 1C → Prof. Tadeu). Junte-o ao `boletim` com `merge` e calcule a média das médias dos alunos de cada professor, em ordem decrescente.

In [ ]:
# Sua resposta — Exercício 8

---
## Bloco C — NumPy

**Exercício 9.** Crie uma matriz 5×5 com os números de 1 a 25. Extraia: a diagonal principal, a soma de cada linha, todos os elementos maiores que 15 e o traço (soma da diagonal).

In [ ]:
# Sua resposta — Exercício 9

**Exercício 10.** Simule 10.000 lançamentos de **dois dados** com `np.random.default_rng`. Qual a probabilidade estimada de a soma ser 7? Compare com o valor teórico (1/6). Bônus: mostre a distribuição das somas com `np.unique(..., return_counts=True)`.

In [ ]:
# Sua resposta — Exercício 10

**Exercício 11.** Dado o array `notas = np.array([4.5, 6.0, 7.5, 9.0, 10.0, 3.0])`, faça a **normalização min-max** para a escala 0–100 (o menor vira 0, o maior vira 100). Depois calcule também o **z-score** de cada nota.

In [ ]:
# Sua resposta — Exercício 11

**Exercício 12.** Usando a série de temperatura limpa `serie` (Seção 4.3), remodele para 7 dias × 24 horas e conte quantas horas por dia ficaram acima de 26 °C. Qual foi o dia mais quente por esse critério?

In [ ]:
# Sua resposta — Exercício 12

---
## Bloco D — Integração

**Exercício 13.** Leia `notas_brutas.txt` (Seção 4.4) diretamente com `pd.read_csv` usando os parâmetros corretos (`sep`, `decimal`, `na_values`). Preencha as notas ausentes com a **média da turma** do aluno (dica: `groupby(...).transform("mean")`) e salve o resultado em `notas_limpas.csv`. Leia o arquivo salvo para conferir.

In [ ]:
# Sua resposta — Exercício 13

<a id="7"></a>
# 7. Gabarito dos exercícios

Existem várias soluções corretas; as apresentadas aqui priorizam clareza.

### Gabarito — Exercício 1

In [ ]:
itens = ["arroz", "feijão", "leite", "pão", "café"]

with open("compras.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(itens) + "\n")

with open("compras.txt", encoding="utf-8") as f:
    for i, item in enumerate(f, start=1):
        print(f"{i}. {item.strip()}")

### Gabarito — Exercício 2

In [ ]:
texto = """A ciência é feita de perguntas.
Os dados ajudam a responder.
Python transforma dados em conhecimento."""

with open("texto.txt", "w", encoding="utf-8") as f:
    f.write(texto)

linhas = palavras = caracteres = 0
with open("texto.txt", encoding="utf-8") as f:
    for linha in f:
        linhas += 1
        palavras += len(linha.split())
        caracteres += len(linha)

print(f"Linhas: {linhas} | Palavras: {palavras} | Caracteres: {caracteres}")

### Gabarito — Exercício 3

In [ ]:
import csv

estados = [
    {"estado": "São Paulo",    "capital": "São Paulo",      "populacao": 44411238},
    {"estado": "Minas Gerais", "capital": "Belo Horizonte", "populacao": 20539989},
    {"estado": "Bahia",        "capital": "Salvador",       "populacao": 14141626},
    {"estado": "Paraná",       "capital": "Curitiba",       "populacao": 11444380},
]

with open("estados.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["estado", "capital", "populacao"])
    w.writeheader()
    w.writerows(estados)

with open("estados.csv", newline="", encoding="utf-8") as f:
    linhas = list(csv.DictReader(f))

mais_populoso = max(linhas, key=lambda l: int(l["populacao"]))   # int() porque tudo vem como string
print("Mais populoso:", mais_populoso["estado"], "-",
      f"{int(mais_populoso['populacao']):,}".replace(",", "."), "habitantes")

### Gabarito — Exercício 4

In [ ]:
import json

prefs = {"tema": "claro", "fonte": 12, "notificacoes": True}
with open("prefs.json", "w", encoding="utf-8") as f:
    json.dump(prefs, f, indent=2, ensure_ascii=False)

with open("prefs.json", encoding="utf-8") as f:
    prefs_lidas = json.load(f)

prefs_lidas["tema"] = "escuro"
prefs_lidas["fonte"] += 2

with open("prefs.json", "w", encoding="utf-8") as f:
    json.dump(prefs_lidas, f, indent=2, ensure_ascii=False)

with open("prefs.json", encoding="utf-8") as f:
    print(f.read())

### Gabarito — Exercício 5

In [ ]:
boletim = pd.read_csv("boletim.csv")
boletim["media"] = boletim[["bim1", "bim2", "bim3"]].mean(axis=1).round(2)

print("(a) Alunos por turma:")
print(boletim["turma"].value_counts().sort_index(), "\n")

print("(b) Média do bim2 na 1B:", round(boletim.loc[boletim["turma"] == "1B", "bim2"].mean(), 2), "\n")

print("(c) Top 3 médias:")
print(boletim.nlargest(3, "media")[["nome", "turma", "media"]])

### Gabarito — Exercício 6

In [ ]:
boletim["faixa_faltas"] = pd.cut(boletim["faltas"], bins=[-1, 3, 8, 100], labels=["Baixa", "Média", "Alta"])
print(boletim[["nome", "faltas", "faixa_faltas"]])
print()
print(boletim["faixa_faltas"].value_counts())

# Alternativa com apply:
# boletim["faixa_faltas"] = boletim["faltas"].apply(lambda f: "Baixa" if f <= 3 else "Média" if f <= 8 else "Alta")

### Gabarito — Exercício 7

In [ ]:
print("(a) Produto mais vendido em quantidade:", vendas.groupby("produto")["quantidade"].sum().idxmax())

nomes_dias = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]
vendas["dia_semana"] = vendas["data"].dt.dayofweek          # 0 = segunda ... 6 = domingo
por_dia_semana = vendas.groupby("dia_semana")["total"].sum().round(2)
por_dia_semana.index = [nomes_dias[i] for i in por_dia_semana.index]

print("\n(b) Faturamento total por dia da semana:")
print(por_dia_semana)

### Gabarito — Exercício 8

In [ ]:
professores = pd.DataFrame({
    "turma": ["1A", "1B", "1C"],
    "professor": ["Prof. Marcos", "Profa. Lúcia", "Prof. Tadeu"],
})
juncao = boletim.merge(professores, on="turma", how="left")
print(juncao.groupby("professor")["media"].mean().round(2).sort_values(ascending=False))

### Gabarito — Exercício 9

In [ ]:
M = np.arange(1, 26).reshape(5, 5)
print(M)
print("Diagonal:", np.diag(M))
print("Soma por linha:", M.sum(axis=1))
print("Maiores que 15:", M[M > 15])
print("Traço:", np.trace(M))

### Gabarito — Exercício 10

In [ ]:
rng = np.random.default_rng(0)
d1 = rng.integers(1, 7, size=10_000)
d2 = rng.integers(1, 7, size=10_000)
soma = d1 + d2

prob_7 = np.mean(soma == 7)          # média de booleanos = proporção de True
print(f"P(soma = 7) simulada: {prob_7:.4f} | teórica: {1/6:.4f}\n")

valores, contagens = np.unique(soma, return_counts=True)
for v, c in zip(valores, contagens):
    print(f"{v:2d}: {'█' * (c // 100)} {c}")

### Gabarito — Exercício 11

In [ ]:
notas = np.array([4.5, 6.0, 7.5, 9.0, 10.0, 3.0])

normalizado = (notas - notas.min()) / (notas.max() - notas.min()) * 100
print("Min-max (0-100):", normalizado.round(1))

z = (notas - notas.mean()) / notas.std()
print("Z-score:        ", z.round(2))

### Gabarito — Exercício 12

In [ ]:
por_dia = serie.to_numpy().reshape(7, 24)        # 'serie' foi criada na Seção 4.3
horas_quentes = (por_dia > 26).sum(axis=1)       # True conta como 1

for dia, h in enumerate(horas_quentes, start=1):
    print(f"Dia {dia}: {h} horas acima de 26 °C")
print("Dia mais quente:", horas_quentes.argmax() + 1)

### Gabarito — Exercício 13

In [ ]:
dados = pd.read_csv("notas_brutas.txt", sep=";", decimal=",", na_values=["N/A"], skipinitialspace=True)
for coluna in ["nome", "turma"]:
    dados[coluna] = dados[coluna].str.strip()
print("Antes:\n", dados, "\n")

# transform devolve uma Series do MESMO tamanho do DataFrame, com a média da turma repetida em cada linha
dados["nota"] = dados["nota"].fillna(dados.groupby("turma")["nota"].transform("mean"))
dados.to_csv("notas_limpas.csv", index=False)

print("Depois (lido de notas_limpas.csv):\n", pd.read_csv("notas_limpas.csv"))

<a id="8"></a>
# 8. Referências e próximos passos

- Documentação oficial do Python — [Leitura e escrita de arquivos](https://docs.python.org/pt-br/3/tutorial/inputoutput.html#reading-and-writing-files), [módulo `csv`](https://docs.python.org/pt-br/3/library/csv.html), [módulo `json`](https://docs.python.org/pt-br/3/library/json.html)
- Pandas — [Guia do usuário](https://pandas.pydata.org/docs/user_guide/index.html) e [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- NumPy — [Guia para iniciantes](https://numpy.org/doc/stable/user/absolute_beginners.html)
- McKinney, W. *Python para Análise de Dados*. Novatec (o autor é o criador do pandas).
- VanderPlas, J. *Python Data Science Handbook*. O'Reilly (gratuito online).

### Sugestões para continuar
1. Refaça a aplicação 4.1 com uma planilha real da sua escola (exportada em CSV).
2. Explore `df.plot()` e a biblioteca **matplotlib**/**seaborn** para visualização.
3. Estude `pd.read_excel` com várias abas (`sheet_name=None`) e `pd.read_html` para tabelas de sites.
4. Aprofunde-se em séries temporais: `resample`, `rolling`, `shift`.

---
*Apostila gerada para estudo autônomo — todas as células são executáveis e os dados são sintéticos.*